# Maps

The purpose of this assignment is to get practice with interactivity using the ipywidgets interface with mappable data.


## A few extra tips & tricks for this assignment:
1. If you are using `contextily` and your map looks off, one thing to try is to pass the `crs` as `crs.to_string()` (see the prep notebook for this week for some examples)
1. If you are using `contextily` and an error is getting thrown about the map server, try a different server with a call like `ctx.add_basemap(ax=ax, source=ctx.providers.OpenStreetMap.Mapnik)`.  See the prep notebook, this <a href="https://github.com/geopandas/contextily/issues/223">github issue</a>, and the `contextily` <a href="https://contextily.readthedocs.io/en/latest/providers_deepdive.html">list of tile providers</a> for more info.

**Please see Homework Prompt in PrairieLearn interface for more details on the requirements for this assignment.**


A rough outline of elements of code and write-up is shown below:

## Interactive Plot

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets

In [2]:
apartments = gpd.read_file("Apartments.geojson")
schools = gpd.read_file("Educational_Properties.geojson")

In [3]:
apartments = apartments.to_crs(epsg=4326)
schools = schools.to_crs(epsg=4326)

In [4]:
apartments["Units"] = pd.to_numeric(apartments["Units"], errors="coerce").fillna(0).astype(int)

building_type_selector = widgets.Dropdown(
    options=["All"] + sorted(apartments["Building_Type"].dropna().unique()),
    value="All",
    description="Building Type:",
)

In [5]:
units_slider = widgets.IntSlider(
    min=int(apartments["Units"].min()),
    max=int(apartments["Units"].max()),
    step=1,
    value=int(apartments["Units"].median()),
    description="Min Units:",
)

In [6]:
def update_map(building_type, min_units):
    fig, ax = plt.subplots(figsize=(10, 7))
    
    filtered_apts = apartments.copy()
    if building_type != "All":
        filtered_apts = filtered_apts[filtered_apts["Building_Type"] == building_type]
    filtered_apts = filtered_apts[filtered_apts["Units"] >= min_units]
    
    filtered_apts.plot(ax=ax, color='blue', alpha=0.5, edgecolor='black')
    schools.plot(ax=ax, color='red', alpha=0.7, edgecolor='black')

    apt_patch = mpatches.Patch(color='blue', alpha=0.5, label="Apartments")
    school_patch = mpatches.Patch(color='red', alpha=0.7, label="Schools")
    ax.legend(handles=[apt_patch, school_patch], loc="upper right")

    ax.set_title("Apartment Complexes and Educational Institutions in Champaign")
    plt.show()

interactive_plot = widgets.interactive(update_map, building_type=building_type_selector, min_units=units_slider)
display(interactive_plot)

interactive(children=(Dropdown(description='Building Type:', options=('All', 'Building', 'Complex', 'Fraternit…

## Write Up

Please include 1-2 paragraphs explaining your visualization and the user interaction that you are implementing:

 * What are you trying to show? 
 * What projection are you using and why?
 * What did you find interesting? 
 * Who do you think would be interested in using this visualization? 

### 1. What am I trying to show?
This interactive map visualizes apartment complexes and educational institutions in Champaign, Illinois. Users can filter apartments by building type and minimum number of units, making it easier to explore housing availability near schools. This can provide insights for students, families, and city planners looking to understand such residential patterns.

By overlaying schools, or educational properties, and apartments, the map helps identify areas with high-density housing near educational institutions. This can assist with decision-making when it comes to housing accessibility, urban development, community planning, and even family planning.

### 2. What Projection am I Using?
This map uses the EPSG:4326 (WGS 84) projection, which represents geographic data using latitude and longitude. It is a widely used projection for web mapping and for displaying location-based data accurately.

I selected this projection based on class materials, where we explored different coordinate reference systems. Since WGS 84 ensures compatibility across multiple mapping platforms, it was a good and safe choice for integrating apartment and school datasets smoothly.

### 3. What Did I Find Interesting?
One interesting aspect of this project was ensuring that both layers aligned correctly by converting them to the same CRS. Initially, seeing the discrepancies between projections made the datasets seem misaligned, so it highlights the importance of using a consistent spatial reference system.

Also, filtering apartments by building type and unit size revealed trends in housing density. Large apartment complexes clustered around certain areas, likely indicating high student populations or planned residential developments.

### 4. Who Would Be Interested?
This map is could be particularly useful for students and families seeking housing near schools, as well as real estate developers looking for areas with high-density apartments. The ability to filter by unit size also helps prospective renters understand housing availability.

Urban planners and city officials can utilize this visualization to assess residential distribution near educational institutions, helping with decisions on zoning, infrastructure, and community development. The interactive nature makes it a practical tool for housing and urban analysis.